In [ ]:
import os
import re
import nltk
from nltk import Tree

import spacy
#spacy.cli.download("en_core_web_sm")
# nlp = spacy.load("en_core_web_sm") 

import pandas as pd
import numpy as np

import textdescriptives as td

from dataset_evaluation.utils import add_column
from dataset_evaluation.evaluation_framework import EvaluationFramework

import folia.main as folia
from pathlib import Path
from collections import defaultdict, Counter
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
from datasets import load_dataset

import pylangacq
import ast
from matplotlib.lines import Line2D

from vendi_score import text_utils

from diversity import (
	compression_ratio
)

import seaborn as sns
from matplotlib.patches import Patch
from datetime import datetime

plt.style.use("seaborn-v0_8-whitegrid")

In [3]:
data = pd.read_csv('/Users/sabijn/Documents/PhD/code/storylm_p1_data/human_eval/prep4qualtrics/questions_cleaned_and_sampled.csv')

In [4]:
ref_standard_dep = Path('datasets/BasiScript/BS_dep_lexicon_unk_newlemmatizer.csv')
ref_standard_uni = Path('datasets/BasiScript/BS_unigram_lexicon.csv')
ref_standard_bi = Path('datasets/BasiScript/BS_bigram_lexicon.csv')

In [5]:
ref_spoken_b_csv = Path('/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/CGN/CGN_pos_bigram.csv')
ref_spoken_u_csv  = Path('/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/CGN/CGN_pos_unigram.csv')
ref_spoken_t_csv = Path('/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/CGN/CGN_pos_trigram.csv')

In [ ]:
eval_f = EvaluationFramework(language='nl',
							  pos_unigram=ref_spoken_u_csv, 
							  pos_bigram=ref_spoken_b_csv,
							  pos_trigram=ref_spoken_t_csv,
							  ref_unigram=ref_standard_uni,
							  ref_bigram=ref_standard_bi,
							  ref_ling_constrained=ref_standard_dep,
							  embedding_model='jegormeister/bert-base-dutch-cased')

In [7]:
# Surprise: creative perplexity
eval_f.add_pipe('creative_perplexity_dep')
# Local contextuality
eval_f.add_pipe("local_contextuality")
# Grammaticality
eval_f.add_pipe('grammaticality')
# Diversity (lexical) self-bleu
eval_f.add_pipe('self-bleu')
# Diversity (lexical) moving mtld
eval_f.add_pipe('lexical_diversity')
# Complexity (lexical) unique words
eval_f.add_pipe('unique-words')
# Complexity (lexical) average word length
eval_f.add_pipe('avg-word-length')
# Complexity (syntactic)  average components
eval_f.add_pipe('average_components')
# Complexity (syntactic) dependency distance
eval_f.add_pipe('dependency_distance')
# Complexity (syntactic) syntactic tree depth
eval_f.add_pipe('syntactic_depth')
# Words before root
eval_f.add_pipe('wbr_average')

In [ ]:
def load_or_run_eval(eval_f, dataset, column, path_name, *, run=False):
	if Path(path_name).exists() and not run:
		df = pd.read_csv(path_name)
		if 'lexical_diversity' in df.columns:
			df['lexical_diversity'] = df['lexical_diversity'].apply(ast.literal_eval)
			
		return df

	dataset = eval_f.run_pipeline_on_df(dataset, column)
	dataset.to_csv(path_name, index=False)
	return dataset

In [9]:
generated_eval_results = load_or_run_eval(eval_f, data, 'story', 'human_eval/automatic_metric_results/eval_results_human_eval_set.csv')

In [16]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

def get_content_words_per_sentence(sent):
	content_words = []
	for token in sent:
		if str(token.pos_) in ['ADV', 'VERB', 'NOUN', 'ADJ']:
			content_words.append(token.lemma_)

	return content_words

def get_content_words_per_story(story):
	content_words_per_sent = []
	doc = nlp(story)
	sent_list = list(doc.sents)

	for sent in sent_list:
		temp = get_content_words_per_sentence(sent)
		if len(temp) != 0:
			content_words_per_sent.append(temp)
	
	return content_words_per_sent

def compute_sem_dist(e1, e2):
	return (1 - cos_sim(e1, e2)).item()

def compute_pairwise_distances(embeddings):
	if len(embeddings) <= 1:
		return [0]

	avg_pairwise_distances = []
	for i in range(len(embeddings)):
		pairwise_distances = []
		for j in range(len(embeddings)):
			if i != j:
				distance = compute_sem_dist(embeddings[i], embeddings[j])
				pairwise_distances.append(distance)
		avg_pairwise_distances.append(np.mean(pairwise_distances))
	return avg_pairwise_distances


def get_embedding(word):
	# only get tokens belonging to the word, exlucing the [CLS] and [SEP] token embeddings
	embeddings = MODEL.encode(word, output_value="token_embeddings")[1:-1]
	
	if embeddings.ndim == 2:
		embeddings = embeddings.mean(axis=0)

	return embeddings

def compute_avg_dist(sent):
	"""
	Average distances between the embeddings of the content words in a sentence
	"""
	embeddings = [get_embedding(word) for word in sent]
	
	return np.mean(compute_pairwise_distances(embeddings))

def compute_surprise(story):
	sentences = get_content_words_per_story(story)

	if len(sentences) <= 1:
		return 0, []
	
	# remove sentences with only one content word
	sentence_avg_distances = [compute_avg_dist(sent) for sent in sentences if len(sent) > 1]
	raw_surprises = [abs(sentence_avg_distances[i] - sentence_avg_distances[i-1]) for i in range(1, len(sentence_avg_distances))]

	# len(raw_surprises) == len(F) - 1
	try:
		return (2 / (len(raw_surprises))) * sum(raw_surprises), raw_surprises
	except:
		return 0, []

# MODEL = SentenceTransformer('jegormeister/bert-base-dutch-cased')
MODEL = SentenceTransformer('NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers')

In [17]:
nlp = spacy.load("nl_core_news_lg") 
generated_eval_results['surprise'] = generated_eval_results.apply(lambda row: compute_surprise(row['story'])[0], axis = 1) 

In [18]:
generated_eval_results.to_csv('human_eval/automatic_metric_results/eval_results_human_eval_set.csv')